# Split set123 into train / internal test / external test

New design:
- set_3 is always external test.
- set_1 + set_2 are split into train=190, internal test=98, and external-from-set12=42.
- external test = external-from-set12 + all set_3.
- Fold values: train folds 0-4, internal test fold 5, external test fold 6.
- The fixed train/internal/external split is shared across all output files.
- The listed random states only change the 5-fold split inside the train set.


In [1]:

import os
import numpy as np
import pandas as pd 

from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold


# ============================================================
# Settings
# ============================================================

patient_list_path = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123.xlsx"
out_root = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists"

clinical_variables_processed_path = os.path.join(
    out_root,
    "image_label_info_set123_clinical_variables_processed.xlsx",
)

task = "prognosis"
label_col = "Prognosis_label"

split_col = "split"
fold_col = "fold"

# Search starts here. If split_mode == "manual", manual_split_random_state is used directly.
split_mode = "manual"   # choose: "search" or "manual"
manual_split_random_state = 106
split_random_state = 100
max_split_search_attempts = 50000
accept_best_if_no_perfect = True

# set_1 + set_2 are split into these three groups.
n_train = 188
n_internal_test = 96
n_external_from_set12 = 46

# Exact positive-count targets for the fixed split.
# set_3 is always external, so external_total_pos_target determines how many
# positive set_1+set_2 cases should be assigned to external test.
# Current target: internal test has 24 positives, external test has 17 positives.
internal_test_pos_target = 24
external_total_pos_target = 17

# set_3 is always external test.
expected_set3_n = 18

# Train fold settings. These random states only affect train fold assignment.
n_splits = 5
internal_test_fold_value = 5
external_test_fold_value = 6
manual_fold_random_state_list = [0, 10, 20, 30, 40]

# Tumor variables used for train/internal/external balance.
tumor_variable_cols = [
    "Tumor_AP_diameter_mm",
    "Tumor_longitudinal_diameter_mm",
    "Tumor_transverse_diameter_mm",
    "Tumor_volume_mm3",
]

tumor_p_alpha = 0.05

# Pairwise match requirement based on significance (p < tumor_p_alpha):
# - internal_test_best_match: required match count between train vs internal test
# - external_test_best_match: required match count between train vs external test
# Max value equals len(tumor_variable_cols). Default external rule is disabled (0).
internal_test_best_match = 3
external_test_best_match = 1

# Output audit table for the selected split and generated fold files.
tumor_balance_report_path = os.path.join(
    out_root,
    f"image_label_info_set123_5fold_{task}_tumor_balance_report.xlsx",
)

print("Patient list:", patient_list_path)
print("Clinical processed table:", clinical_variables_processed_path)
print("Output root:", out_root)
print("Fold random states:", manual_fold_random_state_list)
print("internal_test_best_match:", internal_test_best_match)
print("external_test_best_match:", external_test_best_match)
print("internal_test_pos_target:", internal_test_pos_target)
print("external_total_pos_target:", external_total_pos_target)


Patient list: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123.xlsx
Clinical processed table: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_clinical_variables_processed.xlsx
Output root: /host/e/D/Data/Habitats/Jishuitan/Patient_lists
Fold random states: [0, 10, 20, 30, 40]
internal_test_best_match: 3
external_test_best_match: 1
internal_test_pos_target: 24
external_total_pos_target: 17


In [2]:

# ============================================================
# Load patient table and tumor-size clinical table
# ============================================================

df0 = pd.read_excel(patient_list_path)

print("Loaded:", patient_list_path)
print("Shape:", df0.shape)
print("Columns:", list(df0.columns))

if label_col not in df0.columns:
    raise ValueError(f"Missing label column: {label_col}")

if df0[label_col].isna().any():
    missing_label = df0.loc[df0[label_col].isna(), ["Patient_set", "Patient_index"]]
    display(missing_label)
    raise ValueError(f"{label_col} contains missing values.")

df0[label_col] = df0[label_col].astype(int)
df0["Patient_set"] = df0["Patient_set"].astype(str)
df0["Patient_index"] = df0["Patient_index"].astype(str)

set_counts = df0["Patient_set"].value_counts().sort_index()
print("\nPatient_set counts:")
display(set_counts.rename_axis("Patient_set").reset_index(name="n"))

set12_mask = df0["Patient_set"].isin(["set_1", "set_2"])
set3_mask = df0["Patient_set"] == "set_3"

n_set12 = int(set12_mask.sum())
n_set3 = int(set3_mask.sum())

print("set_1 + set_2 n:", n_set12)
print("set_3 n:", n_set3)

if n_set12 != n_train + n_internal_test + n_external_from_set12:
    raise ValueError(
        f"Expected set1+set2 n={n_train + n_internal_test + n_external_from_set12}, "
        f"but found {n_set12}."
    )

if n_set3 != expected_set3_n:
    raise ValueError(f"Expected set3 n={expected_set3_n}, but found {n_set3}.")

clinical_tumor_df = pd.read_excel(clinical_variables_processed_path)
clinical_tumor_df["Patient_set"] = clinical_tumor_df["Patient_set"].astype(str)
clinical_tumor_df["Patient_index"] = clinical_tumor_df["Patient_index"].astype(str)

required_clinical_cols = ["Patient_set", "Patient_index"] + tumor_variable_cols
missing_clinical_cols = [col for col in required_clinical_cols if col not in clinical_tumor_df.columns]
if missing_clinical_cols:
    raise KeyError(f"Missing columns in clinical processed table: {missing_clinical_cols}")

# Keep tumor features in a separate aligned dataframe. Saved split files remain based on df0.
tumor_aligned_df = df0[["Patient_set", "Patient_index"]].merge(
    clinical_tumor_df[required_clinical_cols],
    on=["Patient_set", "Patient_index"],
    how="left",
    validate="one_to_one",
)

if tumor_aligned_df[tumor_variable_cols].isna().any().any():
    missing_cases = tumor_aligned_df.loc[
        tumor_aligned_df[tumor_variable_cols].isna().any(axis=1),
        ["Patient_set", "Patient_index"] + tumor_variable_cols,
    ]
    display(missing_cases)
    raise RuntimeError("Some cases are missing tumor-size variables.")

tumor_values = tumor_aligned_df[tumor_variable_cols].astype(float)

print("\nOverall label distribution:")
display(
    df0[label_col]
    .value_counts(dropna=False)
    .rename_axis(label_col)
    .reset_index(name="count")
)

print(f"Overall {label_col}=1 fraction: {df0[label_col].mean():.4f}")

print("\nTumor variable table aligned to patient list:")
display(tumor_aligned_df.head())


Loaded: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123.xlsx
Shape: (348, 41)
Columns: ['Patient_set', 'Patient_index', 'Include', 'Have_seg', 'X_shape', 'Y_shape', 'Slice_num', 'Spacing', 'Image_filepath', 'Mask_filepath', 'Medical_record_number', 'Registration_number', 'Prognosis_label', 'Pathologic_label', 'Follow_up_time', 'Admission_time', 'Sort_order', 'Name', 'Pathologic_fracture', 'Length', 'Width', 'Height', 'Sex', 'Side', 'Lesion_site', 'Age', 'Height_at_visit (cm)', 'Weight_at_visit (kg)', 'WBC (*10^9/L)', 'HGB (g/L)', 'PLT (*10^9/L)', 'CRP (mg/L)', 'ALP (IU/L)', 'Total_cholesterol (mmol/L)', 'Triglycerides (mmol/L)', 'LDL (mmol/L)', 'LDH (IU/L)', 'PT (S)', 'APTT (S)', 'Fibrinogen (mg/dL)', 'D-dimer (mg/L FEU)']

Patient_set counts:


,Patient_set,n
0,set_1,99
1,set_2,231
2,set_3,18


set_1 + set_2 n: 330
set_3 n: 18



Overall label distribution:


,Prognosis_label,count
0,0,258
1,1,90


Overall Prognosis_label=1 fraction: 0.2586

Tumor variable table aligned to patient list:


,Patient_set,Patient_index,Tumor_AP_diameter_mm,Tumor_longitudinal_diameter_mm,Tumor_transverse_diameter_mm,Tumor_volume_mm3
0,set_1,1,122.987806,144.968236,116.466989,816281.627888
1,set_1,5,79.927032,89.614892,79.265397,137699.289711
2,set_1,7,90.605079,105.821709,98.946985,267149.414439
3,set_1,8,40.059695,48.504358,41.457716,30504.607393
4,set_1,11,89.836784,89.984684,91.460815,362204.846620


In [3]:

# ============================================================
# Helper functions for split balance and fold assignment
# ============================================================

def add_tumor_columns(df_base):
    """Return a temporary dataframe with tumor variables added by row order."""
    df_tmp = df_base.copy()
    for var in tumor_variable_cols:
        df_tmp[var] = tumor_values[var].to_numpy()
    return df_tmp


def split_group_summary(df_eval):
    summary = (
        df_eval
        .groupby(split_col)[label_col]
        .agg(n="count", positive_count="sum", positive_fraction="mean")
        .reset_index()
    )
    return summary


def set_distribution_summary(df_eval):
    return (
        df_eval
        .groupby([split_col, "Patient_set"])
        .size()
        .reset_index(name="n")
        .sort_values([split_col, "Patient_set"])
    )


def continuous_p_value_for_label(df_group, value_col):
    """Within one split group, compare one tumor variable between label=0 and label=1.

    If both label groups are approximately normal by Shapiro-Wilk, use Welch's t-test.
    Otherwise use Mann-Whitney U test.
    """
    g0 = pd.to_numeric(
        df_group.loc[df_group[label_col] == 0, value_col],
        errors="coerce",
    ).dropna().to_numpy(float)
    g1 = pd.to_numeric(
        df_group.loc[df_group[label_col] == 1, value_col],
        errors="coerce",
    ).dropna().to_numpy(float)

    if len(g0) < 3 or len(g1) < 3:
        return np.nan, "too_few"

    try:
        normal0 = stats.shapiro(g0).pvalue > 0.05 if len(g0) <= 5000 else False
        normal1 = stats.shapiro(g1).pvalue > 0.05 if len(g1) <= 5000 else False
    except Exception:
        normal0 = False
        normal1 = False

    if normal0 and normal1:
        p_value = stats.ttest_ind(g0, g1, equal_var=False, nan_policy="omit").pvalue
        return float(p_value), "Welch_t_test"

    p_value = stats.mannwhitneyu(g0, g1, alternative="two-sided").pvalue
    return float(p_value), "Mann_Whitney_U"


def tumor_p_table_across_splits(df_eval, dataset_name="train_internal_external"):
    """Within each split group, compare label=0 vs label=1 for every tumor variable.

    For each tumor variable, calculate p-values/significance for train, internal test,
    and external test. Then compute pairwise significance matches:
    - train vs internal test
    - train vs external test
    """
    rows = []
    group_order = ["train", "internal test", "external test"]

    for var in tumor_variable_cols:
        row = {
            "Dataset": dataset_name,
            "Variable": var,
        }

        sig_status_by_group = {}

        for group_name in group_order:
            df_group = df_eval[df_eval[split_col] == group_name].copy()
            p_value, method = continuous_p_value_for_label(df_group, var)
            is_sig = bool(pd.notna(p_value) and p_value < tumor_p_alpha)
            sig_status_by_group[group_name] = is_sig

            safe_group_name = group_name.replace(" ", "_")
            row[f"P_{safe_group_name}_label0_vs_label1"] = p_value
            row[f"P_method_{safe_group_name}"] = method
            row[f"Significant_{safe_group_name}_p_lt_0_05"] = is_sig
            row[f"n_{safe_group_name}"] = int(df_group.shape[0])
            row[f"n_label0_{safe_group_name}"] = int((df_group[label_col] == 0).sum())
            row[f"n_label1_{safe_group_name}"] = int((df_group[label_col] == 1).sum())

            for label_value in [0, 1]:
                values = pd.to_numeric(
                    df_group.loc[df_group[label_col] == label_value, var],
                    errors="coerce",
                ).dropna().to_numpy(float)
                row[f"median_label{label_value}_{safe_group_name}"] = float(np.median(values)) if len(values) > 0 else np.nan
                row[f"iqr25_label{label_value}_{safe_group_name}"] = float(np.percentile(values, 25)) if len(values) > 0 else np.nan
                row[f"iqr75_label{label_value}_{safe_group_name}"] = float(np.percentile(values, 75)) if len(values) > 0 else np.nan

        train_sig = sig_status_by_group["train"]
        internal_sig = sig_status_by_group["internal test"]
        external_sig = sig_status_by_group["external test"]

        match_train_internal = (train_sig == internal_sig)
        match_train_external = (train_sig == external_sig)

        row["Match_train_vs_internal_test"] = bool(match_train_internal)
        row["Match_train_vs_external_test"] = bool(match_train_external)
        row["Match_both_pairs"] = bool(match_train_internal and match_train_external)

        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_three_way_split_df(df_candidate):
    df_candidate_with_tumor = add_tumor_columns(df_candidate)
    p_table = tumor_p_table_across_splits(df_candidate_with_tumor)

    internal_match_count = int(p_table["Match_train_vs_internal_test"].sum())
    external_match_count = int(p_table["Match_train_vs_external_test"].sum())

    p_cols = [
        "P_train_label0_vs_label1",
        "P_internal_test_label0_vs_label1",
        "P_external_test_label0_vs_label1",
    ]
    p_values = pd.concat([p_table[col] for col in p_cols], ignore_index=True)
    min_p = float(p_values.min(skipna=True))
    if np.isnan(min_p):
        min_p = -1.0

    return internal_match_count, external_match_count, min_p, p_table, df_candidate_with_tumor


def assign_train_folds(df_split_base, fold_random_state):
    """Assign folds 0-4 to train, fold 5 to internal test, and fold 6 to external test."""
    df_out = df_split_base.copy()
    df_out[fold_col] = -1

    train_mask = df_out[split_col] == "train"
    internal_mask = df_out[split_col] == "internal test"
    external_mask = df_out[split_col] == "external test"

    train_indices = df_out.index[train_mask].to_numpy()
    y_train = df_out.loc[train_indices, label_col].astype(int).to_numpy()

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=fold_random_state,
    )

    for fold_id, (_, val_pos) in enumerate(skf.split(train_indices, y_train)):
        val_indices = train_indices[val_pos]
        df_out.loc[val_indices, fold_col] = fold_id

    df_out.loc[internal_mask, fold_col] = internal_test_fold_value
    df_out.loc[external_mask, fold_col] = external_test_fold_value

    if (df_out[fold_col] < 0).any():
        raise RuntimeError(f"Unassigned fold exists for random_state={fold_random_state}")

    return df_out


In [4]:
# ============================================================
# Fixed train/internal/external split with exact label targets
# and tumor-variable balance
# ============================================================

if split_mode not in ["search", "manual"]:
    raise ValueError(f"split_mode must be 'search' or 'manual'. Got: {split_mode}")

max_match_count = len(tumor_variable_cols)
if internal_test_best_match < 0 or internal_test_best_match > max_match_count:
    raise ValueError(f"internal_test_best_match must be in [0, {max_match_count}]")
if external_test_best_match < 0 or external_test_best_match > max_match_count:
    raise ValueError(f"external_test_best_match must be in [0, {max_match_count}]")

set12_indices = df0.index[set12_mask].to_numpy()
set3_indices = df0.index[set3_mask].to_numpy()

set12_pos_indices = df0.index[set12_mask & (df0[label_col].astype(int) == 1)].to_numpy()
set12_neg_indices = df0.index[set12_mask & (df0[label_col].astype(int) == 0)].to_numpy()

set3_pos_count = int(df0.loc[set3_indices, label_col].astype(int).sum())
set3_neg_count = int((df0.loc[set3_indices, label_col].astype(int) == 0).sum())

external_set12_pos_target = int(external_total_pos_target - set3_pos_count)
external_set12_neg_target = int(n_external_from_set12 - external_set12_pos_target)
internal_test_neg_target = int(n_internal_test - internal_test_pos_target)

train_pos_target = int(len(set12_pos_indices) - external_set12_pos_target - internal_test_pos_target)
train_neg_target = int(len(set12_neg_indices) - external_set12_neg_target - internal_test_neg_target)

if external_set12_pos_target < 0:
    raise ValueError(
        "external_total_pos_target is smaller than the number of fixed positive set_3 cases."
    )

expected_train_n_from_targets = train_pos_target + train_neg_target
if expected_train_n_from_targets != n_train:
    raise ValueError(
        f"Exact label targets imply train n={expected_train_n_from_targets}, expected n_train={n_train}."
    )

print("Exact label-count targets:")
print("  fixed set_3 external:", len(set3_indices), "cases,", set3_pos_count, "positive +", set3_neg_count, "negative")
print("  set_1+set_2 external:", external_set12_pos_target, "positive +", external_set12_neg_target, "negative")
print("  internal test:", internal_test_pos_target, "positive +", internal_test_neg_target, "negative")
print("  train:", train_pos_target, "positive +", train_neg_target, "negative")

selected_split_random_state = None
selected_df_split_base = None
selected_p_table = None
selected_internal_match_count = None
selected_external_match_count = None
selected_min_p = None

best_record = None


def build_three_way_split_for_seed(candidate_seed):
    """Build the fixed split by sampling label-positive and label-negative cases separately.

    Current design:
    - all set_3 cases are assigned to external test
    - set_1+set_2 external test is sampled with an exact positive count
    - internal test is sampled from the remaining set_1+set_2 cases with an exact positive count
    - train receives the remaining set_1+set_2 cases
    """
    rng = np.random.RandomState(candidate_seed)

    external_pos_idx = rng.choice(
        set12_pos_indices,
        size=external_set12_pos_target,
        replace=False,
    )
    remaining_pos_idx = np.setdiff1d(set12_pos_indices, external_pos_idx, assume_unique=False)

    internal_pos_idx = rng.choice(
        remaining_pos_idx,
        size=internal_test_pos_target,
        replace=False,
    )
    train_pos_idx = np.setdiff1d(remaining_pos_idx, internal_pos_idx, assume_unique=False)

    external_neg_idx = rng.choice(
        set12_neg_indices,
        size=external_set12_neg_target,
        replace=False,
    )
    remaining_neg_idx = np.setdiff1d(set12_neg_indices, external_neg_idx, assume_unique=False)

    internal_neg_idx = rng.choice(
        remaining_neg_idx,
        size=internal_test_neg_target,
        replace=False,
    )
    train_neg_idx = np.setdiff1d(remaining_neg_idx, internal_neg_idx, assume_unique=False)

    train_idx = np.concatenate([train_pos_idx, train_neg_idx])
    internal_test_idx = np.concatenate([internal_pos_idx, internal_neg_idx])
    external_from_set12_idx = np.concatenate([external_pos_idx, external_neg_idx])

    df_candidate = df0.copy()
    df_candidate[split_col] = ""

    df_candidate.loc[train_idx, split_col] = "train"
    df_candidate.loc[internal_test_idx, split_col] = "internal test"
    df_candidate.loc[external_from_set12_idx, split_col] = "external test"
    df_candidate.loc[set3_indices, split_col] = "external test"

    if (df_candidate[split_col] == "").any():
        raise RuntimeError("Some cases were not assigned to train/internal/external.")

    split_counts = df_candidate[split_col].value_counts().to_dict()
    if int(split_counts.get("train", 0)) != n_train:
        raise RuntimeError("Train count is inconsistent with exact label targets.")
    if int(split_counts.get("internal test", 0)) != n_internal_test:
        raise RuntimeError("Internal test count is inconsistent with exact label targets.")
    if int(split_counts.get("external test", 0)) != n_external_from_set12 + expected_set3_n:
        raise RuntimeError("External test count is inconsistent with exact label targets.")

    actual_internal_pos = int(df_candidate.loc[df_candidate[split_col] == "internal test", label_col].astype(int).sum())
    actual_external_pos = int(df_candidate.loc[df_candidate[split_col] == "external test", label_col].astype(int).sum())

    if actual_internal_pos != internal_test_pos_target:
        raise RuntimeError(f"Internal test positive count mismatch: {actual_internal_pos} vs {internal_test_pos_target}")
    if actual_external_pos != external_total_pos_target:
        raise RuntimeError(f"External test positive count mismatch: {actual_external_pos} vs {external_total_pos_target}")

    return df_candidate, train_idx, internal_test_idx, external_from_set12_idx


if split_mode == "manual":
    seeds_to_try = [manual_split_random_state]
else:
    seeds_to_try = range(split_random_state, split_random_state + max_split_search_attempts)

print("Searching/using train/internal/external split...")
print(
    "Requirement: exact label-count targets plus pairwise significance match counts must satisfy "
    f"train-vs-internal >= {internal_test_best_match}, "
    f"train-vs-external >= {external_test_best_match}."
)

for attempt_i, candidate_seed in enumerate(seeds_to_try, start=1):
    df_candidate, train_idx, internal_test_idx, external_from_set12_idx = build_three_way_split_for_seed(candidate_seed)
    internal_match_count, external_match_count, min_p, p_table, df_candidate_with_tumor = evaluate_three_way_split_df(df_candidate)

    record = {
        "seed": candidate_seed,
        "internal_match_count": internal_match_count,
        "external_match_count": external_match_count,
        "min_p": min_p,
        "p_table": p_table,
        "df_candidate": df_candidate,
        "train_idx": train_idx,
        "internal_test_idx": internal_test_idx,
        "external_from_set12_idx": external_from_set12_idx,
    }

    if best_record is None:
        best_record = record
    else:
        # Prefer larger internal match count, then larger external match count,
        # then larger minimum p value.
        if (
            (internal_match_count > best_record["internal_match_count"]) or
            (
                internal_match_count == best_record["internal_match_count"] and
                external_match_count > best_record["external_match_count"]
            ) or
            (
                internal_match_count == best_record["internal_match_count"] and
                external_match_count == best_record["external_match_count"] and
                min_p > best_record["min_p"]
            )
        ):
            best_record = record

    match_ok = (
        internal_match_count >= internal_test_best_match and
        external_match_count >= external_test_best_match
    )

    if attempt_i == 1 or attempt_i % 500 == 0 or match_ok:
        print(
            f"  attempt={attempt_i}, seed={candidate_seed}, "
            f"internal_match={internal_match_count}/{max_match_count}, "
            f"external_match={external_match_count}/{max_match_count}, min_p={min_p:.4g}, "
            f"best_internal={best_record['internal_match_count']}, "
            f"best_external={best_record['external_match_count']}, best_min_p={best_record['min_p']:.4g}"
        )

    if match_ok:
        selected_split_random_state = candidate_seed
        selected_df_split_base = df_candidate.copy()
        selected_p_table = p_table.copy()
        selected_internal_match_count = internal_match_count
        selected_external_match_count = external_match_count
        selected_min_p = min_p
        break

if selected_df_split_base is None:
    if not accept_best_if_no_perfect:
        raise RuntimeError(
            "No split satisfied exact label-count and pairwise tumor-variable significance match requirements "
            f"within {max_split_search_attempts} attempts."
        )

    print("\nNo perfect split found. Using best available split because accept_best_if_no_perfect=True.")
    selected_split_random_state = best_record["seed"]
    selected_df_split_base = best_record["df_candidate"].copy()
    selected_p_table = best_record["p_table"].copy()
    selected_internal_match_count = best_record["internal_match_count"]
    selected_external_match_count = best_record["external_match_count"]
    selected_min_p = best_record["min_p"]

split_random_state = selected_split_random_state
manual_split_random_state = selected_split_random_state
df_split_base = selected_df_split_base.copy()

df_split_base_with_tumor = add_tumor_columns(df_split_base)

fixed_train_indices = set(df_split_base.index[df_split_base[split_col] == "train"].tolist())
fixed_internal_test_indices = set(df_split_base.index[df_split_base[split_col] == "internal test"].tolist())
fixed_external_test_indices = set(df_split_base.index[df_split_base[split_col] == "external test"].tolist())

print("\nFixed split created with selected split_random_state =", split_random_state)
print(
    "Selected pairwise match counts:",
    f"train-vs-internal={selected_internal_match_count}/{max_match_count},",
    f"train-vs-external={selected_external_match_count}/{max_match_count}",
)
print("Selected minimum tumor p value:", selected_min_p)

print("\nSplit summary:")
display(split_group_summary(df_split_base))

print("\nSet distribution by split:")
display(set_distribution_summary(df_split_base))

print("\nWithin-split tumor p-values and pairwise match flags:")
display(selected_p_table)

expected_split_counts = {
    "train": n_train,
    "internal test": n_internal_test,
    "external test": n_external_from_set12 + expected_set3_n,
}
actual_counts = df_split_base[split_col].value_counts().to_dict()
for split_name, expected_n in expected_split_counts.items():
    actual_n = int(actual_counts.get(split_name, 0))
    if actual_n != expected_n:
        raise RuntimeError(f"Unexpected {split_name} n: expected {expected_n}, got {actual_n}")

external_set3_n = int(((df_split_base[split_col] == "external test") & (df_split_base["Patient_set"] == "set_3")).sum())
if external_set3_n != expected_set3_n:
    raise RuntimeError(f"Not all set_3 cases are in external test. Found {external_set3_n}/{expected_set3_n}.")


Exact label-count targets:
  fixed set_3 external: 18 cases, 7 positive + 11 negative
  set_1+set_2 external: 10 positive + 36 negative
  internal test: 24 positive + 72 negative
  train: 49 positive + 139 negative
Searching/using train/internal/external split...
Requirement: exact label-count targets plus pairwise significance match counts must satisfy train-vs-internal >= 3, train-vs-external >= 1.
  attempt=1, seed=106, internal_match=3/4, external_match=1/4, min_p=0.0049, best_internal=3, best_external=1, best_min_p=0.0049

Fixed split created with selected split_random_state = 106
Selected pairwise match counts: train-vs-internal=3/4, train-vs-external=1/4
Selected minimum tumor p value: 0.0049002525654440405

Split summary:


,split,n,positive_count,positive_fraction
0,external test,64,17,0.265625
1,internal test,96,24,0.250000
2,train,188,49,0.260638



Set distribution by split:


,split,Patient_set,n
0,external test,set_1,11
1,external test,set_2,35
2,external test,set_3,18
3,internal test,set_1,27
4,internal test,set_2,69
5,train,set_1,61
6,train,set_2,127



Within-split tumor p-values and pairwise match flags:


,Dataset,Variable,P_train_label0_vs_label1,P_method_train,Significant_train_p_lt_0_05,n_train,n_label0_train,n_label1_train,median_label0_train,iqr25_label0_train,...,n_label1_external_test,median_label0_external_test,iqr25_label0_external_test,iqr75_label0_external_test,median_label1_external_test,iqr25_label1_external_test,iqr75_label1_external_test,Match_train_vs_internal_test,Match_train_vs_external_test,Match_both_pairs
0,train_internal_external,Tumor_AP_diameter_mm,0.012669,Mann_Whitney_U,True,188,139,49,73.587741,60.195147,...,17,76.884887,61.800125,92.958369,86.352115,65.146125,98.702150,False,False,False
1,train_internal_external,Tumor_longitudinal_diameter_mm,0.281815,Mann_Whitney_U,False,188,139,49,111.916776,87.553378,...,17,120.000000,93.781296,146.431983,129.488403,111.493934,162.486294,True,True,True
2,train_internal_external,Tumor_transverse_diameter_mm,0.004900,Mann_Whitney_U,True,188,139,49,78.589687,66.651270,...,17,82.924310,69.056618,100.230697,82.622214,67.060228,107.256109,True,False,False
3,train_internal_external,Tumor_volume_mm3,0.009708,Mann_Whitney_U,True,188,139,49,188604.100923,118194.743807,...,17,256612.499298,134063.667188,400515.962088,235000.244583,147837.630788,391463.814870,True,False,False


In [5]:

# ============================================================
# Generate split files from fixed train/internal/external split
# ============================================================

print("Using manually specified train-fold random states:")
print(manual_fold_random_state_list)

saved_paths = []
saved_fold_summary_tables = []

for fold_random_state in manual_fold_random_state_list:
    print("\n============================================================")
    print("Generating:", task, "fold_random_state =", fold_random_state)

    df_out = assign_train_folds(df_split_base, fold_random_state)

    train_mask = df_out[split_col] == "train"
    internal_test_mask = df_out[split_col] == "internal test"
    external_test_mask = df_out[split_col] == "external test"

    if not (df_out.loc[internal_test_mask, fold_col] == internal_test_fold_value).all():
        raise RuntimeError("Internal test fold assignment is not fixed as 5.")

    if not (df_out.loc[external_test_mask, fold_col] == external_test_fold_value).all():
        raise RuntimeError("External test fold assignment is not fixed as 6.")

    if not set(df_out.loc[train_mask, fold_col].unique()).issubset(set(range(n_splits))):
        raise RuntimeError("Train fold assignment contains values outside 0-4.")

    current_train_indices = set(df_out.index[df_out[split_col] == "train"].tolist())
    current_internal_test_indices = set(df_out.index[df_out[split_col] == "internal test"].tolist())
    current_external_test_indices = set(df_out.index[df_out[split_col] == "external test"].tolist())

    if current_train_indices != fixed_train_indices:
        raise RuntimeError("Train split changed unexpectedly.")

    if current_internal_test_indices != fixed_internal_test_indices:
        raise RuntimeError("Internal test split changed unexpectedly.")

    if current_external_test_indices != fixed_external_test_indices:
        raise RuntimeError("External test split changed unexpectedly.")

    print("\nSplit summary:")
    display(split_group_summary(df_out))

    print("\nSet distribution by split:")
    display(set_distribution_summary(df_out))

    print("\nFold summary:")
    fold_summary = (
        df_out
        .groupby([split_col, fold_col])[label_col]
        .agg(n="count", positive_count="sum", positive_fraction="mean")
        .reset_index()
        .sort_values([fold_col, split_col])
    )
    display(fold_summary)

    fold_summary_save = fold_summary.copy()
    fold_summary_save["selected_split_random_state"] = split_random_state
    fold_summary_save["fold_random_state"] = fold_random_state
    saved_fold_summary_tables.append(fold_summary_save)

    out_path = os.path.join(
        out_root,
        f"image_label_info_set123_5fold_{task}_random{fold_random_state}.xlsx",
    )

    df_out.to_excel(out_path, index=False)
    saved_paths.append(out_path)

    print("Saved:", out_path)

print("\nSaved split files:")
for path in saved_paths:
    print(" ", path)


Using manually specified train-fold random states:
[0, 10, 20, 30, 40]

Generating: prognosis fold_random_state = 0

Split summary:


,split,n,positive_count,positive_fraction
0,external test,64,17,0.265625
1,internal test,96,24,0.250000
2,train,188,49,0.260638



Set distribution by split:


,split,Patient_set,n
0,external test,set_1,11
1,external test,set_2,35
2,external test,set_3,18
3,internal test,set_1,27
4,internal test,set_2,69
5,train,set_1,61
6,train,set_2,127



Fold summary:


...


Set distribution by split:


,split,Patient_set,n
0,external test,set_1,11
1,external test,set_2,35
2,external test,set_3,18
3,internal test,set_1,27
4,internal test,set_2,69
5,train,set_1,61
6,train,set_2,127



Fold summary:


,split,fold,n,positive_count,positive_fraction
2,train,0,38,10,0.263158
3,train,1,38,10,0.263158
4,train,2,38,10,0.263158
5,train,3,37,10,0.270270
6,train,4,37,9,0.243243
1,internal test,5,96,24,0.250000
0,external test,6,64,17,0.265625


Saved: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random40.xlsx

Saved split files:
  /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random0.xlsx
  /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random10.xlsx
  /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random20.xlsx
  /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random30.xlsx
  /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random40.xlsx


In [6]:

# ============================================================
# Save tumor balance report
# ============================================================

split_summary_report = split_group_summary(df_split_base)
set_distribution_report = set_distribution_summary(df_split_base)
tumor_p_report = selected_p_table.copy()

fold_summary_report = pd.concat(saved_fold_summary_tables, ignore_index=True)

settings_report = pd.DataFrame([
    {
        "patient_list_path": patient_list_path,
        "clinical_variables_processed_path": clinical_variables_processed_path,
        "task": task,
        "label_col": label_col,
        "selected_split_random_state": split_random_state,
        "split_mode": split_mode,
        "n_train": n_train,
        "n_internal_test": n_internal_test,
        "n_external_from_set12": n_external_from_set12,
        "internal_test_pos_target": internal_test_pos_target,
        "external_total_pos_target": external_total_pos_target,
        "external_set12_pos_target": external_set12_pos_target,
        "external_set12_neg_target": external_set12_neg_target,
        "n_external_total": n_external_from_set12 + expected_set3_n,
        "expected_set3_n": expected_set3_n,
        "fold_random_state_list": str(manual_fold_random_state_list),
        "tumor_p_alpha": tumor_p_alpha,
        "internal_test_best_match": internal_test_best_match,
        "external_test_best_match": external_test_best_match,
        "selected_internal_match_count": selected_internal_match_count,
        "selected_external_match_count": selected_external_match_count,
        "tumor_balance_rule": (
            "Set_3 is fixed as external test. Set_1+set_2 cases are sampled by exact label counts: "
            "external_total_pos_target and internal_test_pos_target are enforced first. "
            "For each tumor variable, compare significance status (p<alpha) pairwise: "
            "train vs internal test, and train vs external test. "
            "Selected split must satisfy internal match count >= internal_test_best_match "
            "and external match count >= external_test_best_match."
        ),
        "selected_minimum_tumor_p_value": selected_min_p,
    }
])

with pd.ExcelWriter(tumor_balance_report_path, engine="openpyxl") as writer:
    settings_report.to_excel(writer, sheet_name="settings", index=False)
    split_summary_report.to_excel(writer, sheet_name="split_summary", index=False)
    set_distribution_report.to_excel(writer, sheet_name="set_distribution", index=False)
    tumor_p_report.to_excel(writer, sheet_name="tumor_p_values", index=False)
    fold_summary_report.to_excel(writer, sheet_name="fold_summary", index=False)

print("Saved tumor balance report:", tumor_balance_report_path)


Saved tumor balance report: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_tumor_balance_report.xlsx


In [7]:

# ============================================================
# Verify saved files
# ============================================================

print("\n============================================================")
print("Verifying saved split files...")

if len(saved_paths) == 0:
    raise RuntimeError("No split files were saved.")

reference_df = pd.read_excel(saved_paths[0])
reference_split = reference_df[split_col].astype(str).tolist()

verify_rows = []

for path in saved_paths:
    df_check = pd.read_excel(path)

    same_split = df_check[split_col].astype(str).tolist() == reference_split
    internal_test_fold_ok = (
        df_check.loc[df_check[split_col] == "internal test", fold_col] == internal_test_fold_value
    ).all()
    external_test_fold_ok = (
        df_check.loc[df_check[split_col] == "external test", fold_col] == external_test_fold_value
    ).all()

    train_n = int((df_check[split_col] == "train").sum())
    internal_test_n = int((df_check[split_col] == "internal test").sum())
    external_test_n = int((df_check[split_col] == "external test").sum())

    external_set3_n = int(((df_check[split_col] == "external test") & (df_check["Patient_set"].astype(str) == "set_3")).sum())

    train_pos_frac = float(df_check.loc[df_check[split_col] == "train", label_col].astype(int).mean())
    internal_test_pos_frac = float(df_check.loc[df_check[split_col] == "internal test", label_col].astype(int).mean())
    external_test_pos_frac = float(df_check.loc[df_check[split_col] == "external test", label_col].astype(int).mean())

    verify_rows.append(
        {
            "file": os.path.basename(path),
            "same_split_as_first_file": same_split,
            "internal_test_fold_is_5": internal_test_fold_ok,
            "external_test_fold_is_6": external_test_fold_ok,
            "train_n": train_n,
            "internal_test_n": internal_test_n,
            "external_test_n": external_test_n,
            "external_set3_n": external_set3_n,
            "train_positive_fraction": train_pos_frac,
            "internal_test_positive_fraction": internal_test_pos_frac,
            "external_test_positive_fraction": external_test_pos_frac,
        }
    )

verify_df = pd.DataFrame(verify_rows)
display(verify_df)

if not verify_df["same_split_as_first_file"].all():
    raise RuntimeError("Not all files have the same fixed train/internal/external split.")

if not verify_df["internal_test_fold_is_5"].all():
    raise RuntimeError("Not all files have internal test fold fixed as 5.")

if not verify_df["external_test_fold_is_6"].all():
    raise RuntimeError("Not all files have external test fold fixed as 6.")

if not (verify_df["train_n"] == n_train).all():
    raise RuntimeError("Train count mismatch in saved files.")

if not (verify_df["internal_test_n"] == n_internal_test).all():
    raise RuntimeError("Internal test count mismatch in saved files.")

if not (verify_df["external_test_n"] == n_external_from_set12 + expected_set3_n).all():
    raise RuntimeError("External test count mismatch in saved files.")

if not (verify_df["external_set3_n"] == expected_set3_n).all():
    raise RuntimeError("Set_3 is not fully assigned to external test in saved files.")

print("All saved files verified.")



Verifying saved split files...


,file,same_split_as_first_file,internal_test_fold_is_5,external_test_fold_is_6,train_n,internal_test_n,external_test_n,external_set3_n,train_positive_fraction,internal_test_positive_fraction,external_test_positive_fraction
0,image_label_info_set123_5fold_prognosis_random...,True,True,True,188,96,64,18,0.260638,0.25,0.265625
1,image_label_info_set123_5fold_prognosis_random...,True,True,True,188,96,64,18,0.260638,0.25,0.265625
2,image_label_info_set123_5fold_prognosis_random...,True,True,True,188,96,64,18,0.260638,0.25,0.265625
3,image_label_info_set123_5fold_prognosis_random...,True,True,True,188,96,64,18,0.260638,0.25,0.265625
4,image_label_info_set123_5fold_prognosis_random...,True,True,True,188,96,64,18,0.260638,0.25,0.265625


All saved files verified.
